# Multi-Agent Systems in Production — AI Lab Instructor Notebook

*LLM & Agentic AI · Expert*



6 tasks, all with solutions.

Runs top to bottom. Each task shows the assignment, then the instructor solution.

In [ ]:
from dataclasses import dataclass
from typing import Any, Callable, Dict, List, Optional, Tuple
import numpy as np
import re


def check(name: str, cond: bool):
    if not cond:
        raise AssertionError(f"Failed: {name}")
    print(f"OK: {name}")

rng = np.random.default_rng(0)


## Task 1: Dataset & Setup

# Multi-Agent Systems in Production — Student Lab

Offline lab: build a small planner / worker / critic system with shared state, guardrails, and evaluation.

You will implement:

- a small tool interface
- a planner that turns tasks into tool calls
- a worker that executes the plan under budget + allowlist guardrails
- a critic that verifies the output
- an orchestrator that ties the roles together

## 0 — Tools + shared state

In [ ]:
from dataclasses import dataclass
from typing import Any, Callable, Dict, List, Optional, Tuple
import numpy as np
import re


def check(name: str, cond: bool):
    if not cond:
        raise AssertionError(f"Failed: {name}")
    print(f"OK: {name}")

rng = np.random.default_rng(0)

**Why this works.** # Multi-Agent Systems in Production — Instructor Lab

Offline lab: build a small planner / worker / critic system with shared state, guardrails, and evaluation.

## 0 — Tools + shared state

## Task 2: 0 — Tools + shared state

## 0 — Tools + shared state

Implement the basic tool layer and a shared state object.

Requirements:

- each tool returns a dictionary like `{"result": ...}`
- the shared state tracks `budget`, `cost`, `memory`, and a `trace`
- every significant event should be logged into the trace

In [ ]:
$10

In [ ]:
# Checks
state = SharedState(budget=2.0, cost=0.0, memory={}, trace=[])
log(state, 'system', 'boot', ok=True)
check('toolbox_n', len(TOOLBOX) >= 4)
check('trace_logs', len(state.trace) == 1)

## Task 3: 1 — Planner agent

## 1 — Planner agent

Implement a deterministic planner that converts the user task into a short sequence of tool calls.

Handle these patterns:

- `(a + b) * c`
- `country of CITY`
- `lower then country of CITY`

The planner should output structured `ToolCall` objects, not the final answer.

In [ ]:
$11

In [ ]:
# Checks
plan_math = planner('(2 + 3) * 4')
check('plan_len', len(plan_math) == 2)
check('plan_last_ref', plan_math[1].args['a'] == '$last')

## Task 4: 2 — Worker agent (exec + guardrails)

## 2 — Worker agent (exec + guardrails)

Implement the worker that executes a plan.

Guardrails to enforce:

- only allowed tools may run
- tool cost must stay within the remaining budget
- `$last` should resolve to the previous tool result
- each execution step should be logged into shared state

In [ ]:
$12

In [ ]:
# Checks
state = SharedState(budget=4.0, cost=0.0, memory={}, trace=[])
out = worker(planner('(2 + 3) * 4'), state)
check('worker_math', out.get('result') == 20)

state_tight = SharedState(budget=1.0, cost=0.0, memory={}, trace=[])
blocked = worker(planner('(2 + 3) * 4'), state_tight)
check('worker_budget_blocked', 'error' in blocked and blocked['error'] == 'budget_exceeded')

## Task 5: 3 — Critic agent (verification / escalation)

## 3 — Critic agent (verification / escalation)

Implement a deterministic critic.

The critic should:

- reject outputs that contain an error
- verify arithmetic tasks exactly
- verify `country of ...` tasks against the known mapping
- return a verdict dictionary such as `{'verdict': 'pass'}` or `{'verdict': 'fail', 'reason': ...}`

In [ ]:
$13

In [ ]:
# Checks
check('critic_pass', critic('(2 + 3) * 4', {'result': 20})['verdict'] == 'pass')
check('critic_fail', critic('country of Tokyo', {'result': 'tokyo'})['verdict'] == 'fail')

## Task 6: 4 — Orchestrator + evaluation

## 4 — Orchestrator + evaluation

Implement the top-level orchestrator.

It should:

- create fresh shared state
- ask the planner for a plan
- run the worker
- ask the critic to verify the result
- return a compact `RunResult`

Then evaluate the system on a few tasks and confirm the budget guardrail still works.

In [ ]:
$14

In [ ]:
# Checks
oks = 0
for t, expected in TASKS:
    r = orchestrate(t, budget=4.0)
    ok = r.ok and (r.answer == expected)
    oks += int(ok)

success = oks / len(TASKS)
check('good_success', success == 1.0)

r_budget = orchestrate('(2 + 3) * 4', budget=1.0)
check('budget_fail', not r_budget.ok and r_budget.reason == 'budget_exceeded')